#### Import des librairies

In [1]:
import pandas as pd
from tqdm import tqdm
import numpy as np

In [3]:
path_file = '~/Bureau/exports/20241112/AGS_20241112_exports_agronomes_-archive/AGS_20241112_exports_agronomes_assolees_synthetisees.csv'

ENTREPOT_PATH = '~/Bureau/utils/data/'
df = {}

#### Import des données

In [4]:
# ----------------------------- #
# IMPORT DES DONNÉES DATAGROSYST#
# ----------------------------- #


def import_df(df_name, path_data, sep, index_col=None):
    df[df_name] = pd.read_csv(path_data+df_name+'.csv', sep = sep, index_col=index_col, low_memory=False).replace({'\r\n': '\n'}, regex=True)

def import_dfs(df_names, path_data, sep = ',', index_col=None, verbose=False):
    for df_name in tqdm(df_names) : 
        if(verbose) :
            print(" - ", df_name)
        import_df(df_name, path_data, sep, index_col=index_col)

tables_with_id = [
    'recolte_rendement_prix', 
    'destination_valorisation',
    'action_realise_agrege', 
    'action_synthetise_agrege'
]

tables_without_id = [
]

# import des données de l'entrepôt avec la colonne 'id' en index 
import_dfs(tables_with_id, ENTREPOT_PATH, sep = ',', index_col='id', verbose=False)

# import des données du magasin
import_dfs(tables_without_id, ENTREPOT_PATH, sep = ',', verbose=False)

100%|██████████| 4/4 [00:33<00:00,  8.38s/it]
0it [00:00, ?it/s]


In [6]:
studied_realise_ids = [
    'fr.inra.agrosyst.api.entities.effective.EffectiveCropCycleNode_49d34aeb-4faa-4148-bf97-695d2bc19104',
    'fr.inra.agrosyst.api.entities.effective.EffectiveCropCycleNode_273b56df-fb85-4aa9-8722-b5ff19e8e95d',
    'fr.inra.agrosyst.api.entities.effective.EffectiveCropCycleNode_77b7434f-5361-4e3f-8913-87d013fb0238' 
]

studied_synthetise_ids = [
    'fr.inra.agrosyst.api.entities.practiced.PracticedCropCycleConnection_ebb0bb5e-8d8c-4f37-bf0f-620bcd625783',
    'fr.inra.agrosyst.api.entities.practiced.PracticedCropCycleConnection_7bcbbf3d-7dda-4f1a-985d-b573f39454e4',
    'fr.inra.agrosyst.api.entities.practiced.PracticedCropCycleConnection_1d10c59d-ac38-4c91-9332-5c6db59c8104'
]

In [22]:
df['action_realise_agrege_test'] = df['action_realise_agrege'].loc[
    df['action_realise_agrege']['noeuds_realise_id'].isin(studied_realise_ids)
]

df['action_synthetise_agrege_test'] = df['action_synthetise_agrege'].loc[
    df['action_synthetise_agrege']['connection_synthetise_id'].isin(studied_synthetise_ids)
]

df['recolte_rendement_prix_test'] = df['recolte_rendement_prix'].loc[
   (df['recolte_rendement_prix']['action_id'].isin(df['action_realise_agrege_test'].index)) | 
   (df['recolte_rendement_prix']['action_id'].isin(df['action_synthetise_agrege_test'].index))
]

df['destination_valorisation_test'] = df['destination_valorisation'].loc[
    df['destination_valorisation'].index.isin(df['recolte_rendement_prix_test']['destination_id'])
]

In [23]:
path='./'
df['action_realise_agrege_test'].to_csv(path+'action_realise_agrege'+'.csv')
df['action_synthetise_agrege_test'].to_csv(path+'action_synthetise_agrege'+'.csv')
df['destination_valorisation_test'].to_csv(path+'destination_valorisation'+'.csv')
df['recolte_rendement_prix_test'].to_csv(path+'recolte_rendement_prix'+'.csv')